In [ ]:
tools = [
    {
        "toolSpec": {
            "name": "print_sentiment_scores",
            "description": "Prints the sentiment scores of a given text."
            "inputSchema": {
                "json": {
                    "type": "object"
                    "properties": {
                        "positive_score": {"type": "number", "description": "The positive sentiment score, ranging from 0.0 to 1.0."},
                        "negative_score": {"type": "number", "description": "The negative sentiment score, ranging from 0.0 to 1.0."},
                        "neutral_score": {"type": "number", "description": "The neutral sentiment score, ranging from 0.0 to 1.0."},
                    },
                    "required": ["positive_score", "negative_score", "neutral_score"]
                }
            }
        }
    }
]

In [ ]:
import json
def analyze_sentiment(content):

    query = f"""
    <text>
    {content}
    </text>

    Only use the print_sentiment_scores tool.
    """

    messages = [{
        "role": "user",
        "content": [{"text": query}]
    }]

    inference_config={"maxTokens":400}
    tool_config = {"tools":tools}

    # Send the message.
    response = bedrock_client.converse(
        modelId=MODEL_ID,
        messages=messages,
        inferenceConfig=inference_config,
        toolConfig=tool_config
    )

    json_sentiment= None
    for content in response["output"]["message"]["content"]
        if content.get("toolUse") is not None and content["toolUse"]["name"] == "print_sentiment_scores":
            json_sentiment = content["toolUse"]["input"]
            break

    if json_sentiment:
        print("Sentiment Aanalysis (JSON):")
        print(json.dumps(json_sentiment, indent=2))
    else:
        print("No sentiment analysis found in the response.")

In [ ]:
analyze_sentiment("I hate having to do the laundry!")

## Forcing Structured Outputs

In [ ]:
tools = [
    {
        "toolSpec": {
            "name": "print_entities",
            "description": "Prints extract named entities."
            "inputSchema": {
                "json": {
                    "type": "object"
                    "properties": {
                        "entities": {
                            "type": "array",
                            "items": {
                                "properties": {
                                    "name": {"type": "string", "description": "The extracted entity name."},
                                    "type": {"type": "string", "description": "The entity type (e.g., PERSON, ORGANIZATION, LOCATION)."},
                                    "context": {"type": "string", "description": "The context in which the entity appears in the text."},
                                },
                                "required": ["name","type","context"]
                            },
                        }
                    },
                    "required": ["entities"]
                }
            }
        }
    }
]

text = "John works at Amazon in Seattle. He met with Sarah, the CEB of Acme Inc., last week in San Francisco."

query = f"""
<document>
{text}
</document>

Use the print_entities tool.
"""


messages = [
    {
        "role": "user",
        "content": [{"text": query}]
    }
]

inference_config={"maxTokens":400}
tool_config={"tools":tools,"tool":{"name":"print_entities"}}

# Send the message.
response = bedrock_client.converse(
    modelId=MODEL_ID,
    messages=messages,
    inferenceConfig=inference_config,
    toolConfig=tool_config
)
response["output"]